# LipSyncNet - Model Architecture

This notebook documents the architecture design and construction plan for **LipSyncNet** (LSN), a lip-reading model combining a 3D-CNN frontend, EfficientNetB0 feature extraction, and a Bi-LSTM backend with CTC decoding.

We propose a modified model based on LSN with three temporal variants - BiLSTM, No Temporal, and Transformer Encoding.

**NOTES**  
[1] The paper reports total=307,595,340 / trainable=304,824,800 / frozen=2,780,531.

- Frozen-param count (2,780,531) does not correspond to any stage boundary in torchvision EfficientNet-B0.  
  Actual cumulative counts:
  - stages 0-5 frozen -> 851,808
  - stages 0-6 frozen -> 2,878,156 <- closest to paper's 2,780,531
  - stages 0-7 frozen -> 3,595,388

  No boundary equals 2,780,531 exactly.  The paper was built in TensorFlow/Keras, whose EfficientNet weight layout differs slightly from torchvision's.  We use stages 0-6 frozen (2,878,156), which is the closest achievable clean boundary and matches the original code's behavior.

- LSTM parameter counts in Table 1 are conflicting with the stated architecture.  A PyTorch BiLSTM(input=70912, hidden=512) has ~292M parameters, not the paper's 5,971,968.

  Reverse-engineering:
  - 5,971,968 / 2 dirs = 2,985,984 per direction
  - Solving 4*(inp*512 + 512^2 + 2*512) = 2,985,984  ->  inp ≈ 944
  - No architecturally meaningful input dimension produces this.
  - The LSTM-2 value (10,485,760) back-solves to inp=2,046 (hid=512) or inp=254 (hid=1,024) — neither matches the stated architecture.

[2] Regarding Self-Attention:  
Figure 9 (block diagram) shows a self-attention layer between Bi-LSTM-2 and the output. Table 1 has no corresponding row, states no configuration (heads, d_model, dropout), and the cumulative parameter count in Table 1 leaves no room for it. See code block for more context.

In [1]:
from __future__ import annotations

import math
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import time
import string
# from google.colab import drive
import os
import shutil


In [2]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO = "ranro1/lipsyncnet-checkpoints"  # ← update with your username/repo

hf_api = HfApi(token=HF_TOKEN)
print("HF ready")

HF ready


In [3]:
class Conv3DBlock(nn.Module):
    """Single conv3d + BN + ReLU + MaxPool block."""

    def __init__(self,
                 in_channels:  int,
                 out_channels: int,
                 kernel_size:  tuple,
                 padding:      tuple,
                 pool_kernel:  tuple = (1, 2, 2),
                 pool_stride:  tuple = (1, 2, 2)):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels,
                              kernel_size=kernel_size,
                              padding=padding,
                              bias=False)
        self.bn   = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool3d(kernel_size=pool_kernel, stride=pool_stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pool(self.relu(self.bn(self.conv(x))))

In [4]:
class Frontend3DCNN(nn.Module):
    """
    Four-block 3D-CNN front-end (Table 1).

    Input : (B, 1, T, 46, 140)
    Output: (B, T, 8192) -- 512 * 2 * 8 per timestep

    Block  filters  kernel    padding   pool(1,2,2)   spatial out
    ──────────────────────────────────────────────────────────────
      1      64    (3,5,5)   (1,2,2)    (1,2,2)       23 * 70
      2     128    (3,5,5)   (1,2,2)    (1,2,2)       11 * 35
      3     256    (3,3,3)   (1,1,1)    (1,2,2)        5 * 17
      4     512    (3,3,3)   (1,1,1)    (1,2,2)        2 *  8     -> flat 8,192
    """

    OUT_C    = 512
    OUT_H    = 2
    OUT_W    = 8
    FLAT_DIM = OUT_C * OUT_H * OUT_W   # 8,192

    def __init__(self):
        super().__init__()
        self.block1 = Conv3DBlock(1,   64,  (3, 5, 5), (1, 2, 2))
        self.block2 = Conv3DBlock(64,  128, (3, 5, 5), (1, 2, 2))
        self.block3 = Conv3DBlock(128, 256, (3, 3, 3), (1, 1, 1))
        self.block4 = Conv3DBlock(256, 512, (3, 3, 3), (1, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)                          # (B,  64, T, 23, 70)
        x = self.block2(x)                          # (B, 128, T, 11, 35)
        x = self.block3(x)                          # (B, 256, T,  5, 17)
        x = self.block4(x)                          # (B, 512, T,  2,  8)
        B, C, T, Hf, Wf = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous()  # (B, T, C, Hf, Wf)
        return x.view(B, T, C * Hf * Wf)           # (B, T, 8192)


In [5]:
class EfficientNet(nn.Module):
    """
    EfficientNet-B0 feature extractor, applied frame-by-frame.

    Input : (B*T, 1, H, W) -- raw grayscale lip frames (NOT 3D-CNN output)
    Output: (B*T, 62720)   -- flattened spatial map (1280 * 7 * 7)

    Freezing policy [D1]:
        Freeze features[0..6] -> 2,878,156 frozen params.
        Unfreeze features[7,8] -> trained end-to-end.

        The paper states 2,780,531 frozen, which no torchvision stage boundary produces.
        The 97,625-param discrepancy is attributed to TF/Keras vs. torchvision weight layout differences.
    """

    FEATURE_DIM = 7 * 7 * 1280   # 62,720

    def __init__(self, freeze_early: bool = True):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features   # indices 0-8; output (B, 1280, 7, 7)

        if freeze_early:
            # Freeze all parameters first
            for param in self.features.parameters():
                param.requires_grad = False
            # Unfreeze stages 7 and 8 (last two MBConv blocks + head)
            for stage_idx in (7, 8):
                for param in self.features[stage_idx].parameters():
                    param.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.expand(-1, 3, -1, -1)                            # (B*T, 3, H, W)
        x = F.interpolate(x, size=(224, 224),
                          mode="bilinear", align_corners=False)  # (B*T, 3, 224, 224)
        x = self.features(x)                                    # (B*T, 1280, 7, 7)
        return torch.flatten(x, 1)                              # (B*T, 62720)

In [6]:
class BiLSTMBackend(nn.Module):
    """
    2* Bi-LSTM with input projection and dropout.
    Used inside LipSyncNetVariant — NOT inside LipSyncNetPaper.

    input_dim -> Linear(1024) -> BiLSTM(512*2) -> Dropout -> BiLSTM(512*2) -> Dropout
    Output dim: 1,024.
    """

    def __init__(self, input_dim: int, hidden: int = 512, dropout: float = 0.5):
        super().__init__()
        self.out_dim    = hidden * 2
        self.input_proj = nn.Linear(input_dim, self.out_dim)
        self.lstm1      = nn.LSTM(self.out_dim, hidden, batch_first=True,
                                  bidirectional=True)
        self.drop1      = nn.Dropout(dropout)
        self.lstm2      = nn.LSTM(self.out_dim, hidden, batch_first=True,
                                  bidirectional=True)
        self.drop2      = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, _ = self.lstm1(self.input_proj(x))
        x     = self.drop1(x)
        x, _ = self.lstm2(x)
        return self.drop2(x)


class IdentityBackend(nn.Module):
    """
    No temporal modeling — ablation baseline.
    Passes fused features directly to the classifier unchanged.
    Isolates the contribution of the temporal backend.
    """

    def __init__(self, input_dim: int, **_):
        super().__init__()
        self.out_dim = input_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x


class _SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding ("Attention Is All You Need" paper; Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 75, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                        * (-math.log(10_000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerBackend(nn.Module):
    """
    2-layer bidirectional Transformer encoder.
    Linear projection -> sinusoidal PE -> 2* TransformerEncoderLayer.
    Output dim: d_model (default 1,024).

    NOTE: max_len in _SinusoidalPE must be >= the longest sequence T in the dataset;
    Update if LRS2 clips exceed 75 frames (see TODO-PRE-5).
    """

    def __init__(self, input_dim: int, d_model: int = 1024,
                 nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.out_dim    = d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc    = _SinusoidalPE(d_model, dropout=dropout)
        self.encoder    = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                       dim_feedforward=d_model * 4,
                                       dropout=dropout, batch_first=True),
            num_layers=num_layers,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(self.pos_enc(self.input_proj(x)))


_BACKEND_REGISTRY: dict[str, type] = {
    "bilstm":      BiLSTMBackend,
    "identity":    IdentityBackend,
    "transformer": TransformerBackend,
}


In [7]:
class SelfAttentionBlock(nn.Module):
    """
    Post-norm multi-head self-attention block placed between Bi-LSTM-2 and
    the classifier, as shown in Figure 9 of the paper.

    Design rationale (all choices forced by context; none are free):

    embed_dim = 1024
        Forced by the Bi-LSTM-2 output shape (B, T, 1024) and the classifier
        input shape (B, T, 1024).  The block must be dimensionality-preserving.

    num_heads = 8
        8 heads * 128 head_dim = 1024.

    Residual connection  (out = LayerNorm(x + Dropout(MHA(x))))
        Standard post-norm formulation.  A bare MHA call with no residual
        would discard positional information accumulated by the LSTMs and
        is not used anywhere in the sequence modelling literature at this
        position.  The paper does not specify, so the conventional choice
        is made and documented.

    dropout = 0.1
        The paper states no dropout for this layer.  0.1 is the standard
        attention dropout used in the original Transformer (Vaswani et al.).
        Keeping it low (vs. the 0.5 used in the LSTM stack) because attention
        weights are already an implicit regulariser. It is configurable.

    No feedforward sublayer
        The paper's block diagram shows a single "Self-Attention" box,
        use TransformerBackend instead during ablation.
    """

    def __init__(self, embed_dim: int = 1024, num_heads: int = 8,
                 dropout: float = 0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(embed_dim=embed_dim,
                                           num_heads=num_heads,
                                           dropout=dropout,
                                           batch_first=True)
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor | None = None) -> torch.Tensor:
        # query = key = value = x  (self-attention over the temporal dimension)
        attn_out, _ = self.attn(x, x, x,
                                key_padding_mask=key_padding_mask,
                                need_weights=False)
        # Post-norm residual:  LayerNorm(x + Dropout(Attn(x)))
        return self.norm(x + self.drop(attn_out))              # (B, T, 1024)


In [8]:
class LipSyncNetPaper(nn.Module):
    """
    Re-implementation of LipSyncNet (Table 1 / Figure 8 & 9).

    Explicit decisions for all ambiguities:
      [1] EfficientNet: stages 0-6 frozen (2,878,156 params; closest to paper).
      [2] Self-attention included (Figure 9), implemented as SelfAttentionBlock
           (embed_dim=1024, num_heads=8).  Absent from Table 1.
           Controlled by `use_self_attn` flag so the no-attention variant (matching Table 1) can still be instantiated for ablation.
      [3] No input projection before LSTM-1; raw 70,912-dim concat fed directly.
    """

    CNN_DIM   = Frontend3DCNN.FLAT_DIM              # 8,192
    EFF_DIM   = EfficientNet.FEATURE_DIM      # 62,720
    FUSED_DIM = CNN_DIM + EFF_DIM                   # 70,912

    def __init__(self,
                 vocab_size:          int  = 40,
                 freeze_early_effnet: bool = True,
                 use_self_attn:       bool = True):
        """
        Args:
            vocab_size          : output classes excluding CTC blank.
            freeze_early_effnet : freeze EfficientNet stages 0-6.
            use_self_attn       : include SelfAttentionBlock after Bi-LSTM-2.
        """
        super().__init__()
        self.vocab_size    = vocab_size
        self.use_self_attn = use_self_attn

        self.cnn3d        = Frontend3DCNN()
        self.efficientnet = EfficientNet(freeze_early=freeze_early_effnet)

        self.lstm1 = nn.LSTM(self.FUSED_DIM, 512, batch_first=True,
                             bidirectional=True)
        self.drop1 = nn.Dropout(0.5)
        self.lstm2 = nn.LSTM(1024, 512, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.5)

        if use_self_attn:
            self.self_attn = SelfAttentionBlock(embed_dim=1024, num_heads=8,
                                                dropout=0.1)

        self.classifier = nn.Linear(1024, vocab_size + 1)  # +1 = CTC blank

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor | None = None) -> torch.Tensor:
        B, T, H, W = x.shape

        # 3D-CNN branch (spatiotemporal)
        cnn_out = self.cnn3d(x.unsqueeze(1))           # (B, T, 8192)

        # EfficientNet branch (per-frame, on raw input)
        frames  = x.view(B * T, 1, H, W)               # (B*T, 1, 46, 140)
        eff_out = self.efficientnet(frames)             # (B*T, 62720)
        eff_out = eff_out.view(B, T, self.EFF_DIM)     # (B, T, 62720)

        fused = torch.cat([cnn_out, eff_out], dim=-1)  # (B, T, 70912)

        # Bi-LSTM back-end
        out, _ = self.lstm1(fused)                     # (B, T, 1024)
        out     = self.drop1(out)
        out, _ = self.lstm2(out)                       # (B, T, 1024)
        out     = self.drop2(out)

        # Skipped when use_self_attn=False.
        if self.use_self_attn:
            out = self.self_attn(out, key_padding_mask=key_padding_mask)
                                                       # (B, T, 1024)
        # Classifier
        logits    = self.classifier(out)               # (B, T, vocab+1)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.permute(1, 0, 2)              # (T, B, vocab+1)


In [9]:
class LipSyncNetVariant(nn.Module):
    """
    Backend selected at construction from {"bilstm", "transformer", "identity"}.

    Note: All backends receive the raw FUSED_DIM (70,912) vector for fair comparison.
    The BiLSTM variant adds a projection (Linear 70912 -> 1024) before the LSTMs;
    this differs slightly from LipSyncNetPaper (i.e., no projection).
    For paper: Both should be reported with the distinction made explicit.
    """

    CNN_DIM   = Frontend3DCNN.FLAT_DIM          # 8,192
    EFF_DIM   = EfficientNet.FEATURE_DIM  # 62,720
    FUSED_DIM = CNN_DIM + EFF_DIM               # 70,912

    def __init__(self,
                 backend:             str  = "bilstm",
                 vocab_size:          int  = 40,
                 freeze_early_effnet: bool = True,
                 **backend_kwargs):
        super().__init__()

        if backend not in _BACKEND_REGISTRY:
            raise ValueError(
                f"Unknown backend '{backend}'. "
                f"Valid options: {list(_BACKEND_REGISTRY)}")

        self.vocab_size   = vocab_size
        self.cnn3d        = Frontend3DCNN()
        self.efficientnet = EfficientNet(freeze_early=freeze_early_effnet)
        self.backend      = _BACKEND_REGISTRY[backend](
                                input_dim=self.FUSED_DIM, **backend_kwargs)
        self.classifier   = nn.Linear(self.backend.out_dim, vocab_size + 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, H, W = x.shape

        cnn_out = self.cnn3d(x.unsqueeze(1))

        frames  = x.view(B * T, 1, H, W)

        eff_out = self.efficientnet(frames).view(B, T, self.EFF_DIM)

        fused   = torch.cat([cnn_out, eff_out], dim=-1)   # (B, T, 70912)

        out       = self.backend(fused)

        logits    = self.classifier(out)
        log_probs = F.log_softmax(logits, dim=-1)

        return log_probs.permute(1, 0, 2)                 # (T, B, vocab+1)

In [10]:
def build_paper_model(vocab_size:    int  = 40,
                      use_self_attn: bool = False,
                      device:        str  = "cpu") -> LipSyncNetPaper:
    """Instantiate the paper-exact LipSyncNetPaper model."""
    return LipSyncNetPaper(vocab_size=vocab_size,
                           use_self_attn=use_self_attn).to(device)


def build_variant(backend:    str = "bilstm",
                  vocab_size: int = 40,
                  device:     str = "cpu",
                  **backend_kwargs) -> LipSyncNetVariant:
    """
    Instantiate a variant model with the chosen temporal backend.
    """
    return LipSyncNetVariant(backend=backend,
                             vocab_size=vocab_size,
                             **backend_kwargs).to(device)


def count_parameters(model: nn.Module) -> dict[str, int]:
    """Return total, trainable, and frozen parameter counts."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}

In [11]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device : {device}\n")

    dummy = torch.randn(2, 75, 46, 140).to(device)

    # Paper model
    paper = build_paper_model(device=device)
    ps    = count_parameters(paper)
    with torch.no_grad():
        po = paper(dummy)
    print(f"[paper]              output={tuple(po.shape)}")
    print(f"                     total={ps['total']:>12,}  "
          f"trainable={ps['trainable']:>12,}  frozen={ps['frozen']:>10,}")
    print(f"                     paper: total=307,595,340  "
          f"trainable=304,824,800  frozen=2,780,531")
    print()

    # Three variants
    for backend in ("bilstm", "transformer", "identity"):
        m  = build_variant(backend=backend, device=device)
        st = count_parameters(m)
        with torch.no_grad():
            o = m(dummy)
        print(f"[variant: {backend:<12}] output={tuple(o.shape)}  "
              f"total={st['total']:>12,}  "
              f"trainable={st['trainable']:>12,}  "
              f"frozen={st['frozen']:>10,}")

Device : cuda

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 136MB/s] 


[paper]              output=(75, 2, 41)
                     total= 307,954,917  trainable= 305,076,761  frozen= 2,878,156
                     paper: total=307,595,340  trainable=304,824,800  frozen=2,780,531

[variant: bilstm      ] output=(75, 2, 41)  total=  94,308,581  trainable=  91,430,425  frozen= 2,878,156
[variant: transformer ] output=(75, 2, 41)  total= 106,901,733  trainable= 104,023,577  frozen= 2,878,156
[variant: identity    ] output=(75, 2, 41)  total=  11,959,781  trainable=   9,081,625  frozen= 2,878,156



# TODO: PREPROCESSING

## TODO-PRE-1  [GRID — paper-subset split for exact reproduction]
- Reproduce the paper's 450 train / 550 test split from a ~1,000 video subset.  The paper does not state which speaker(s) or random seed.
- Adopt seed=42, sample from a single speaker, and document the choice.
- This run targets 96.7% accuracy / 8.2% WER.  Report honestly — do NOT tune the split to match the paper's numbers post-hoc.

## TODO-PRE-2  [GRID — full-dataset split]
- After the paper-subset reproduction, define a speaker-independent split:
- 3 speakers for test, 2 for val, 28 for train.  Report on this split in a separate table clearly labelled "GRID-full".

## TODO-PRE-3  [LRS2 — access]
- Use the official pre-train / train / val / test splits verbatim.
- Do NOT resplit; published baselines use these exact splits.

## TODO-PRE-4  [LRS2 — variable-length handling]
- LRS2 clips vary in length; the model currently assumes fixed T=75.
- Steps:
  - Compute the length distribution over the training set. Choose MAX_T = 95th-percentile clip length (avoids memory blow-up from outliers while preserving most content).
  - Pad shorter clips to MAX_T with zero frames; record true lengths.
  - Pass true lengths as `input_lengths` to CTCLoss — padding frames must NOT contribute to the CTC loss.
  - Update _SinusoidalPE(max_len=MAX_T) in TransformerBackend if MAX_T > 75.
  - 3D-CNN is fully convolutional in T and handles any T >= 3.

## TODO-PRE-5  [ROI extraction — verify consistency with paper]
The preprocessing script must:
- Use dlib 68-point landmarks to crop the lip region.
- Output exactly (H=46, W=140) — landscape orientation, H < W.
- Convert to grayscale before normalization.
- Normalize PER VIDEO: subtract video mean, divide by video std. (NOT per-dataset)
- Save pre-cropped tensors to disk (e.g., .npy or .pt) so ROI detection runs once, not every epoch.

## TODO-PRE-6  [Vocabulary — verify label mapping before ANY training]
- Paper vocabulary: 40 symbols = {space, a-z, ?, !, ', 1-9}
- Index 0    : CTC blank (PyTorch CTCLoss convention)
- Index 1-40 : the 40 symbols in a fixed, documented order
- Write a unit test: encode a known sentence -> decode -> assert round-trip. A mismatch here produces plausible-looking loss curves but wrong text.

# TODO-PRE-8  [Alignment files — silence handling]
- GRID .align files mark silence as 'sil'.  The paper excludes silence.
- Verify preprocessing strips 'sil' tokens and that all remaining label sequences are non-empty for every retained sample.

# TODO-PRE-9  [Data augmentation — training split only]
Paper mentions augmentation without specifying transforms.
- Implement and apply ONLY during training (never val or test):
- Horizontal flip (p=0.5)
- Random time crop: sample T consecutive frames when clip > T frames
- Brightness/contrast jitter: ±20%
- For LRS2, also consider: random rectangular erasing over lip area (p=0.1)?



# TODO: TRAINING

## TODO-TRN-1  [CTCLoss setup]
- loss_fn = torch.nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
- zero_infinity=True suppresses -inf on degenerate short sequences and prevents NaN gradients in early training — essential for stability.

- Call signature:
	- loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

- For fixed-T GRID: input_lengths = torch.full((B,), T, dtype=torch.long)

## TODO-TRN-2  [Gradient clipping]
- Bi-LSTMs on CTC are prone to explosion; the paper is silent on clipping.
- After loss.backward(), before optimizer.step(): grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
- Log grad_norm each epoch; if it regularly saturates at 5.0, lower max_norm.

## TODO-TRN-3  [Batch size — unreported by paper; must be chosen and documented]
- The paper model has ~292M parameters in LSTM-1 alone and is very memory heavy.  Start with batch_size=2 on a single GPU. Use gradient accumulation to reach an effective batch of 8-16:
- loss = loss / accumulation_steps # accumulate for N steps, then: optimizer.step(); optimizer.zero_grad()
- Report batch_size and effective_batch in all experiment tables.

## TODO-TRN-4  [Training harness — single unified entry point]
- Build train.py with argparse flags:
* `--model` - {paper, bilstm, transformer, identity}
* `--dataset` - {grid_subset, grid_full, lrs2}
* `--epochs` - int - (default 100, matching paper)
* `--lr`- float (default 1e-4, fixed — no LR schedule per paper)
* `--batch` - int
* `--accum` - int - (gradient accumulation steps, default 1)
* `--seed` - int - (default 42)
* `--resume` - str - (path to checkpoint, optional)
- All four model conditions MUST use identical hyperparameters for a valid comparison.

## TODO-TRN-5  [Checkpointing]
- Save every 10 epochs AND whenever val_loss improves.
- Checkpoint dict must contain:
	- {epoch, model_state_dict, optimizer_state_dict, val_loss, val_wer, config}, where config captures all argparse arguments so checkpoints are self-describing and runs are fully reproducible from a checkpoint.

## TODO-TRN-6  [Logging]
- Log per epoch: train_loss, val_loss, val_WER, grad_norm, epoch_time_sec.
- Use torch.utils.tensorboard.SummaryWriter or wandb.
- Export loss-vs-epoch curves as static images for the capstone report (reproducing Figures 13 and 14 style from the paper?)

## TODO-TRN-7  [LRS2 training strategy — decide before starting]
LRS2 labelled train split is small (~1,166 utterances); from-scratch training will likely underfit.

Suggestion:
- Phase 1: Train on GRID-full for 100 epochs -> save best checkpoint.
- Phase 2: Load GRID checkpoint; fine-tune on LRS2.  Optionally, freeze 3D-CNN for first 10 LRS2 epochs, then unfreeze.
- Document the chosen strategy explicitly; report which GRID epoch was used to initialize.

## TODO-TRN-8  [Reproducibility — mandatory before any training run]
At the top of train.py:
- import random, numpy as np
- random.seed(seed); np.random.seed(seed)
- torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
- torch.backends.cudnn.deterministic = True
- torch.backends.cudnn.benchmark-   = False

Log and include in capstone appendix:
- Python version, PyTorch version, CUDA version,
- torchvision version, GPU model and VRAM.


## Training Setup

In [12]:
# SETUP Configuration
NUM_FRAMES      = 75                # temporal depth of each video clip
FRAME_H         = 46                # frame height (pixels)
FRAME_W         = 140               # frame width  (pixels)
IN_CHANNELS     = 1                 # grayscale input

VOCAB_SIZE      = 40                # distinct characters
BLANK_INDEX     = 0                 # CTC blank token index (PyTorch CTCLoss convention: blank=0)
NUM_CLASSES     = VOCAB_SIZE + 1    # 41 output units (incl. blank)

EFFICIENTNET_H  = 224
EFFICIENTNET_W  = 224

LSTM_UNITS      = 512               # units per direction; 512×2 = 1024 output
DROPOUT_RATE    = 0.5
LEARNING_RATE   = 1e-4
BEAM_WIDTH      = 100

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
# grid labels are lowercase english words + spaces
VOCAB = [' '] + list(string.ascii_lowercase)

# ids start at 1 (0 is reserved for ctc blank)
char_to_idx = {ch: i + 1 for i, ch in enumerate(VOCAB)}
idx_to_char = {i + 1: ch for i, ch in enumerate(VOCAB)}

VOCAB_SIZE = len(VOCAB)
NUM_CLASSES = VOCAB_SIZE + 1  # includes blank

print("vocab size:", VOCAB_SIZE)
print("num classes:", NUM_CLASSES)
print("blank index:", BLANK_INDEX)
print("sample mapping:", {k: char_to_idx[k] for k in [' ', 'a', 'b', 'z']})

vocab size: 27
num classes: 28
blank index: 0
sample mapping: {' ': 1, 'a': 2, 'b': 3, 'z': 27}


In [14]:
def encode_text(text: str) -> torch.Tensor:
    text = text.lower().strip()
    ids = [char_to_idx[ch] for ch in text]
    return torch.tensor(ids, dtype=torch.long)


def decode_ids(ids) -> str:
    chars = []
    for idx in ids:
        idx = int(idx)
        if idx == BLANK_INDEX:
            continue
        chars.append(idx_to_char[idx])
    return ''.join(chars)

In [15]:
class GridLipReadingDatasetParquet(Dataset):
    def __init__(self, npz_paths):
        self.npz_paths = [str(p) for p in npz_paths]

    def __len__(self):
        return len(self.npz_paths)

    def __getitem__(self, idx):
        path = self.npz_paths[idx]

        data = np.load(path, allow_pickle=True)

        frames = data["frames"]          # expected: (75, 46, 140, 1)
        text = str(data["label"]).lower().strip()

        # remove trailing channel dimension if it is 1
        if frames.ndim == 4 and frames.shape[-1] == 1:
            frames = frames[..., 0]

        frames = torch.tensor(frames, dtype=torch.float32)
        target = encode_text(text)
        target_length = len(target)

        sample = {
            "frames": frames,                 # (75, 46, 140)
            "text": text,
            "target": target,                 # (target_len,)
            "target_length": target_length,
            "path": path
        }

        return sample

class GridLipReadingDataset(Dataset):
    """
    Loads preprocessed .npz files.

    Expected .npz keys:
        - "frames": numpy array, shape (75, 46, 140) or (75, 46, 140, 1)
        - "label":  string (the spoken text)

    If your .npz uses different key names, update FRAME_KEY and LABEL_KEY below.
    """
    FRAME_KEY = "frames"   # ← change if your npz uses a different key
    LABEL_KEY = "label"    # ← change if your npz uses a different key (e.g., "text")

    def __init__(self, npz_paths):
        self.paths = [str(p) for p in npz_paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        data = np.load(path, allow_pickle=True)

        frames = data[self.FRAME_KEY]          # (75, 46, 140) or (75, 46, 140, 1)
        text = str(data[self.LABEL_KEY]).lower().strip()

        # Remove trailing channel dim if present
        if frames.ndim == 4 and frames.shape[-1] == 1:
            frames = frames[..., 0]            # (75, 46, 140)

        frames = torch.tensor(frames, dtype=torch.float32)
        target = encode_text(text)
        target_length = len(target)

        return {
            "frames": frames,                  # (75, 46, 140)
            "text": text,
            "target": target,                  # (target_len,)
            "target_length": target_length,
            "path": path
        }

In [16]:
# # reorganize filders script (to {speaker}->{files})
# import shutil

# """
# Current structure:
#   grid_processed_new/
#     s6_srbh3n.npz
#     s6_srbbzs.npz
#     s8_bbar6a.npz
#     ...

# Target structure:
#   grid_processed_new/
#     s6/
#       s6_srbh3n.npz
#       s6_srbbzs.npz
#     s8/
#       s8_bbar6a.npz
#     ...
# """

# def reorganize_by_speaker(data_dir: str, dry_run: bool = True):
#     """
#     Move .npz files into speaker subfolders based on filename prefix.

#     Filename convention: {speaker}_{utterance_id}.npz
#     Speaker ID = everything before the first underscore (e.g., 's6', 's8', 's12').

#     Args:
#         data_dir:  Path to flat folder containing all .npz files
#         dry_run:   If True, only print what would happen. Set False to execute.
#     """
#     data_path = Path(data_dir)
#     npz_files = sorted(data_path.glob("*.npz"))

#     if not npz_files:
#         print(f"No .npz files found in {data_dir}")
#         return

#     print(f"Found {len(npz_files)} .npz files")

#     speaker_counts = {}

#     for f in npz_files:
#         # Extract speaker ID: everything before the first underscore
#         speaker_id = f.stem.split("_")[0]
#         speaker_dir = data_path / speaker_id
#         dest = speaker_dir / f.name

#         speaker_counts[speaker_id] = speaker_counts.get(speaker_id, 0) + 1

#         if dry_run:
#             continue

#         speaker_dir.mkdir(exist_ok=True)
#         shutil.move(str(f), str(dest))

#     # Summary
#     print(f"\nSpeakers found: {len(speaker_counts)}")
#     for sid in sorted(speaker_counts, key=lambda x: int(x[1:])):
#         print(f"  {sid}: {speaker_counts[sid]} files")

#     if dry_run:
#         print("\n[DRY RUN] No files moved. Set dry_run=False to execute.")
#     else:
#         print(f"\nDone. Files reorganized into {len(speaker_counts)} speaker folders.")


# if __name__ == "__main__":
#     # ── UPDATE THIS PATH ──
#     DATA_DIR = "/content/drive/MyDrive/LSN_Data/grid_processed_new"

#     reorganize_by_speaker(DATA_DIR, dry_run=False)

In [17]:
import os
base = "/kaggle/input/datasets/runnscream/grid-data/grid_processed_new"
print(sorted(os.listdir(base)))

['s1', 's2', 's3', 's4', 's5', 's6']


In [18]:
# sample parquet file loading
# drive.mount('/content/drive')


# -------- UNCOMMENT FROM HERE FOR PARQUET
# sample_path = "/content/drive/MyDrive/LSN_Data/grid_processed/s8_bbar6a.pt"
# obj = torch.load(sample_path, map_location="cpu")


# print("keys:", obj.keys())
# for k, v in obj.items():
#     if isinstance(v, torch.Tensor):
#         print(f"{k}: tensor shape={tuple(v.shape)}, dtype={v.dtype}")
#     else:
#          print(f"{k}: type={type(v)}, value={v}")


# sample .npz file loading
# -------- UNCOMMENT FROM HERE FOR .NPZ
# sample_path = "/content/drive/MyDrive/LSN_Data/grid_processed_new/s6/s6_srbh3n.npz"
sample_path = "/kaggle/input/datasets/runnscream/grid-data/grid_processed_new/s1/s1_bbaf2n.npz"
data = np.load(sample_path, allow_pickle=True)

print("Keys:", list(data.keys()))
for k in data.keys():
    v = data[k]
    if hasattr(v, 'shape'):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: type={type(v)}, value={v}")

Keys: ['frames', 'label']
  frames: shape=(75, 46, 140, 1), dtype=float32
  label: shape=(), dtype=<U21


In [19]:
# # dividing preprocessed files to valid (75 frames) and unvalid (<> 75 frames)
# RAW_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed")
# VALID_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed_valid_75")
# INVALID_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed_invalid")

# VALID_DIR.mkdir(parents=True, exist_ok=True)
# INVALID_DIR.mkdir(parents=True, exist_ok=True)


# def inspect_grid_pt_file(path: Path):
#     """
#     returns:
#         is_valid: bool
#         reason: str
#         meta: dict
#     """
#     try:
#         obj = torch.load(path, map_location="cpu")

#         if not isinstance(obj, dict):
#             return False, "not a dict", {}

#         if "frames" not in obj:
#             return False, "missing 'frames' key", {}

#         if "text" not in obj:
#             return False, "missing 'text' key", {}

#         frames = obj["frames"]
#         text = obj["text"]

#         if not isinstance(frames, torch.Tensor):
#             return False, "'frames' is not a tensor", {}

#         if frames.ndim != 3:
#             return False, f"frames.ndim={frames.ndim}, expected 3", {
#                 "shape": tuple(frames.shape)
#             }

#         if tuple(frames.shape) != (75, 46, 140):
#             return False, f"bad shape {tuple(frames.shape)}", {
#                 "shape": tuple(frames.shape)
#             }

#         if not isinstance(text, str):
#             return False, "'text' is not a string", {}

#         text = text.strip().lower()
#         if len(text) == 0:
#             return False, "empty text", {}

#         return True, "ok", {
#             "shape": tuple(frames.shape),
#             "dtype": str(frames.dtype),
#             "text": text
#         }

#     except Exception as e:
#         return False, f"load error: {type(e).__name__}: {e}", {}


# def split_grid_pt_files(
#     raw_dir: Path,
#     valid_dir: Path,
#     invalid_dir: Path,
#     move_files: bool = False
# ):
#     pt_paths = sorted(raw_dir.glob("*.pt"))

#     valid_paths = []
#     invalid_paths = []
#     invalid_reasons = []

#     action = shutil.move if move_files else shutil.copy2

#     for src_path in pt_paths:
#         is_valid, reason, meta = inspect_grid_pt_file(src_path)

#         if is_valid:
#             dst_path = valid_dir / src_path.name
#             if not dst_path.exists():
#                 action(src_path, dst_path)
#             valid_paths.append(dst_path)
#         else:
#             dst_path = invalid_dir / src_path.name
#             if not dst_path.exists():
#                 action(src_path, dst_path)
#             invalid_paths.append(dst_path)
#             invalid_reasons.append((src_path.name, reason, meta))

#     print(f"total files   : {len(pt_paths)}")
#     print(f"valid files   : {len(valid_paths)}")
#     print(f"invalid files : {len(invalid_paths)}")

#     if len(invalid_reasons) > 0:
#         print("\nfirst invalid examples:")
#         for name, reason, meta in invalid_reasons[:10]:
#             print(f"- {name}: {reason} | meta={meta}")

#     return valid_paths, invalid_paths, invalid_reasons

# valid_paths, invalid_paths, invalid_reasons = split_grid_pt_files(
#     raw_dir=RAW_DIR,
#     valid_dir=VALID_DIR,
#     invalid_dir=INVALID_DIR,
#     move_files=False  # safer: copy first, do not move originals yet
# )

In [20]:
# # validate Dataset

# DATA_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed")
# pt_paths = sorted(DATA_DIR.glob("*.pt"))

# print("num files:", len(pt_paths))
# print("first file:", pt_paths[0])

# dataset = GridLipReadingDataset(pt_paths)

# sample = dataset[0]

# print("path         :", sample["path"])
# print("text         :", sample["text"])
# print("frames shape :", tuple(sample["frames"].shape))
# print("frames dtype :", sample["frames"].dtype)
# print("target       :", sample["target"].tolist())
# print("target length:", sample["target_length"])
# print("decoded      :", decode_ids(sample["target"]))

# print("sample valid file  :", valid_paths[0] if len(valid_paths) > 0 else "none")
# print("sample invalid file:", invalid_paths[0] if len(invalid_paths) > 0 else "none")

# sample_valid = torch.load(valid_paths[0], map_location="cpu")
# print(sample_valid.keys())
# print(sample_valid["frames"].shape)
# print(sample_valid["text"])

# DATA_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed_valid_75")
# pt_paths = sorted(DATA_DIR.glob("*.pt"))

# print("num valid files:", len(pt_paths))
# print("first valid file:", pt_paths[0] if len(pt_paths) > 0 else "none")

In [21]:
# count speakers files
from collections import Counter

# DATA_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed_new")
DATA_DIR = Path("/kaggle/input/datasets/runnscream/grid-data/grid_processed_new")

npz_paths = sorted(DATA_DIR.glob("*/*.npz"))   # speaker_dir/*.npz

print("num files:", len(npz_paths))
print("first file:", npz_paths[0] if npz_paths else "none")

speaker_counts = Counter(p.parent.name for p in npz_paths)
for sid in sorted(speaker_counts, key=lambda x: int(x[1:])):
    print(f"  {sid}: {speaker_counts[sid]}")

num files: 5893
first file: /kaggle/input/datasets/runnscream/grid-data/grid_processed_new/s1/s1_bbaf2n.npz
  s1: 1000
  s2: 1000
  s3: 1000
  s4: 1000
  s5: 1000
  s6: 893


In [22]:
# collate function - preparing the results for the CTC function
def grid_collate_fn(batch):
    frames = torch.stack([item["frames"] for item in batch], dim=0)
    targets = torch.cat([item["target"] for item in batch], dim=0)

    target_lengths = torch.tensor(
        [item["target_length"] for item in batch],
        dtype=torch.long
    )

    input_lengths = torch.full(
        size=(len(batch),),
        fill_value=frames.shape[1],   # 75 frames (LSN paper)
        dtype=torch.long
    )

    texts = [item["text"] for item in batch]
    paths = [item["path"] for item in batch]

    return {
        "frames": frames,                  # (B, 75, 46, 140)
        "targets": targets,                # (sum target lengths,)
        "target_lengths": target_lengths,  # (B,)
        "input_lengths": input_lengths,    # (B,)
        "texts": texts,
        "paths": paths
    }

In [23]:
# validate Dataset

# DATA_DIR = Path("/content/drive/MyDrive/LSN_Data/grid_processed_new")
DATA_DIR = Path("/kaggle/input/datasets/runnscream/grid-data/grid_processed_new")
npz_paths = sorted(DATA_DIR.glob("*/*.npz"))  # speaker_dir/*.npz

print("num files:", len(npz_paths))
print("first file:", npz_paths[0])

dataset = GridLipReadingDataset(npz_paths)

sample = dataset[0]

print("path         :", sample["path"])
print("text         :", sample["text"])
print("frames shape :", tuple(sample["frames"].shape))
print("frames dtype :", sample["frames"].dtype)
print("target       :", sample["target"].tolist())
print("target length:", sample["target_length"])
print("decoded      :", decode_ids(sample["target"]))

num files: 5893
first file: /kaggle/input/datasets/runnscream/grid-data/grid_processed_new/s1/s1_bbaf2n.npz
path         : /kaggle/input/datasets/runnscream/grid-data/grid_processed_new/s1/s1_bbaf2n.npz
text         : bin blue at f two now
frames shape : (75, 46, 140)
frames dtype : torch.float32
target       : [3, 10, 15, 1, 3, 13, 22, 6, 1, 2, 21, 1, 7, 1, 21, 24, 16, 1, 15, 16, 24]
target length: 21
decoded      : bin blue at f two now


In [24]:
# Dataloader
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=grid_collate_fn
)

In [25]:
# split into train / val - PARQUET
# ------------------------ UNCOMMENT FROM HERE
# import random

# def split_train_val(paths, val_ratio=0.2, seed=42):
# # 55-45 split
#     paths = list(paths)
#     rng = random.Random(seed)
#     rng.shuffle(paths)

#     n_val = max(1, int(len(paths) * val_ratio))
#     val_paths = paths[:n_val]
#     train_paths = paths[n_val:]

#     return train_paths, val_paths

# train_paths, val_paths = split_train_val(pt_paths, val_ratio=0.45, seed=42)

# print("train files:", len(train_paths))
# print("val files  :", len(val_paths))


# split into train / val - NPZ
# ------------------------ UNCOMMENT FROM HERE
from collections import defaultdict
import random

def create_paper_split(npz_paths, speakers=None, samples_per_speaker=200,
                       train_size=450, seed=42):
    """
    Reproduce the paper's 1,000-sample / 450-550 split with balanced
    speaker representation.

    Per speaker: 90 train, 110 test (= 450/5, 550/5).
    """
    from collections import defaultdict

    speaker_files = defaultdict(list)
    for p in npz_paths:
        sid = p.parent.name
        if speakers and sid not in speakers:
            continue
        speaker_files[sid].append(p)

    if speakers is None:
        speakers = sorted(speaker_files.keys(), key=lambda x: int(x[1:]))

    n_speakers = len(speakers)
    train_per_speaker = train_size // n_speakers        # 90
    test_per_speaker = samples_per_speaker - train_per_speaker  # 110

    rng = random.Random(seed)
    train_paths = []
    test_paths = []

    for sid in sorted(speakers, key=lambda x: int(x[1:])):
        files = sorted(speaker_files[sid])
        assert len(files) >= samples_per_speaker, \
            f"{sid} has {len(files)} files, need {samples_per_speaker}"

        sampled = rng.sample(files, samples_per_speaker)
        rng.shuffle(sampled)

        train_paths.extend(sampled[:train_per_speaker])
        test_paths.extend(sampled[train_per_speaker:])

    # Final shuffle so batches aren't grouped by speaker
    rng.shuffle(train_paths)
    rng.shuffle(test_paths)

    print(f"Speakers     : {sorted(speakers, key=lambda x: int(x[1:]))}")
    print(f"Per speaker  : {samples_per_speaker} total, {train_per_speaker} train, {test_per_speaker} test")
    print(f"Train total  : {len(train_paths)}")
    print(f"Test total   : {len(test_paths)}")

    return train_paths, test_paths


train_paths, val_paths = create_paper_split(
    npz_paths,
    speakers=['s1', 's2', 's3', 's4', 's5'],
    samples_per_speaker=200,
    train_size=450,
    seed=42
)

Speakers     : ['s1', 's2', 's3', 's4', 's5']
Per speaker  : 200 total, 90 train, 110 test
Train total  : 450
Test total   : 550


## Training Loop

In [26]:
import shutil
def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, save_path):
    """Save to local /tmp first, then copy to Drive (atomic-ish)."""
    tmp_path = f"/tmp/checkpoint_tmp.pt"
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
    }
    torch.save(checkpoint, tmp_path)
    shutil.copy2(tmp_path, save_path)

In [27]:
def load_checkpoint(model, optimizer, load_path, device):
    """Load a saved checkpoint."""
    checkpoint = torch.load(load_path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    epoch = checkpoint["epoch"]
    train_loss = checkpoint.get("train_loss", None)
    val_loss = checkpoint.get("val_loss", None)

    print(f"Loaded checkpoint: epoch={epoch}, val_loss={val_loss:.4f}")
    return model, optimizer, epoch, train_loss, val_loss

In [28]:
paper = build_paper_model(device=device)

ctc_loss_fn = nn.CTCLoss(
    blank=BLANK_INDEX,
    reduction="mean",
    zero_infinity=True
)

optimizer = torch.optim.Adam(
    [p for p in paper.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

MAX_GRAD_NORM = 5.0

In [29]:
def train_one_step(model, batch, optimizer, loss_fn, device, max_grad_norm=None):
    model.train()

    frames = batch["frames"].to(device)
    targets = batch["targets"].to(device)
    input_lengths = batch["input_lengths"].to(device)
    target_lengths = batch["target_lengths"].to(device)

    optimizer.zero_grad(set_to_none=True)

    log_probs = model(frames)
    loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

    loss.backward()

    grad_norm = None
    if max_grad_norm is not None:
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_grad_norm
        ).item()

    optimizer.step()

    return {
        "loss": loss.item(),
        "grad_norm": grad_norm
    }

In [30]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, loss_fn, device, max_grad_norm=None):
    model.train()

    running_loss = 0.0
    running_grad_norm = 0.0
    num_batches = 0

    start_time = time.time()

    for batch in tqdm(loader, desc="Training", leave=False):
        frames = batch["frames"].to(device)
        targets = batch["targets"].to(device)
        input_lengths = batch["input_lengths"].to(device)
        target_lengths = batch["target_lengths"].to(device)

        optimizer.zero_grad(set_to_none=True)

        log_probs = model(frames)
        loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

        loss.backward()

        grad_norm = 0.0
        if max_grad_norm is not None:
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_grad_norm
            ).item()

        optimizer.step()

        running_loss += loss.item()
        running_grad_norm += grad_norm
        num_batches += 1

    epoch_time = time.time() - start_time

    return {
        "loss": running_loss / max(num_batches, 1),
        "grad_norm": running_grad_norm / max(num_batches, 1),
        "time_sec": epoch_time
    }

In [31]:
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    num_batches = 0

    start_time = time.time()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", leave=False):
            frames = batch["frames"].to(device)
            targets = batch["targets"].to(device)
            input_lengths = batch["input_lengths"].to(device)
            target_lengths = batch["target_lengths"].to(device)

            log_probs = model(frames)
            loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

            running_loss += loss.item()
            num_batches += 1

    epoch_time = time.time() - start_time

    return {
        "loss": running_loss / max(num_batches, 1),
        "time_sec": epoch_time
    }

In [32]:
train_dataset = GridLipReadingDataset(train_paths)
val_dataset = GridLipReadingDataset(val_paths)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=grid_collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=grid_collate_fn,
    num_workers=2,
    pin_memory=True
)

print("train batches:", len(train_loader))
print("val batches  :", len(val_loader))

train batches: 225
val batches  : 275


In [33]:
# 1 batch test
optimizer = torch.optim.Adam(
    [p for p in paper.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

MAX_GRAD_NORM = 5.0

batch = next(iter(train_loader))

for step in range(30):
    stats = train_one_step(
        model=paper,
        batch=batch,
        optimizer=optimizer,
        loss_fn=ctc_loss_fn,
        device=device,
        max_grad_norm=MAX_GRAD_NORM
    )

    print(
        f"step={step + 1} | "
        f"loss={stats['loss']:.4f} | "
        f"grad_norm={stats['grad_norm']:.4f}"
    )

step=1 | loss=7.3597 | grad_norm=5.7157
step=2 | loss=6.6829 | grad_norm=5.9465
step=3 | loss=6.0306 | grad_norm=6.7208
step=4 | loss=5.2426 | grad_norm=7.8227
step=5 | loss=4.5340 | grad_norm=8.0809
step=6 | loss=3.9147 | grad_norm=7.0893
step=7 | loss=3.4670 | grad_norm=4.7667
step=8 | loss=3.2526 | grad_norm=2.5293
step=9 | loss=3.2235 | grad_norm=3.6474
step=10 | loss=3.2158 | grad_norm=4.9423
step=11 | loss=3.1726 | grad_norm=5.7570
step=12 | loss=3.0615 | grad_norm=5.2672
step=13 | loss=2.9457 | grad_norm=4.4366
step=14 | loss=2.8435 | grad_norm=3.2515
step=15 | loss=2.7908 | grad_norm=1.9944
step=16 | loss=2.7452 | grad_norm=1.7545
step=17 | loss=2.7400 | grad_norm=2.5627
step=18 | loss=2.7095 | grad_norm=3.5876
step=19 | loss=2.6906 | grad_norm=3.2969
step=20 | loss=2.6522 | grad_norm=3.2662
step=21 | loss=2.6109 | grad_norm=2.8393
step=22 | loss=2.5708 | grad_norm=1.8140
step=23 | loss=2.5407 | grad_norm=1.2977
step=24 | loss=2.5228 | grad_norm=1.1830
step=25 | loss=2.4775 | g

In [34]:
# test one batch from the data_loader
batch = next(iter(train_loader))

frames = batch["frames"].to(device)

paper.eval()
with torch.no_grad():
  log_probs = paper(frames)

print("input: ", tuple(frames.shape))
print("output shape: ", tuple(log_probs.shape))
print("output dtype: ", log_probs.dtype)
print("output device: ", log_probs.device)

input:  (2, 75, 46, 140)
output shape:  (75, 2, 41)
output dtype:  torch.float32
output device:  cuda:0


In [35]:
# ##### 1 epoch test
# # Fresh model + optimizer for the 1-epoch test
# paper = build_paper_model(device=device)

# optimizer = torch.optim.Adam(
#     [p for p in paper.parameters() if p.requires_grad],
#     lr=LEARNING_RATE
# )

# # 1 epoch train
# train_stats = train_one_epoch(
#     model=paper,
#     loader=train_loader,
#     optimizer=optimizer,
#     loss_fn=ctc_loss_fn,
#     device=device,
#     max_grad_norm=MAX_GRAD_NORM
# )

# print(
#     f"TRAIN | loss={train_stats['loss']:.4f} | "
#     f"grad_norm={train_stats['grad_norm']:.4f} | "
#     f"time={train_stats['time_sec']:.1f}s"
# )

# # 1 epoch val
# val_stats = validate_one_epoch(
#     model=paper,
#     loader=val_loader,
#     loss_fn=ctc_loss_fn,
#     device=device
# )

# print(
#     f"VAL   | loss={val_stats['loss']:.4f} | "
#     f"time={val_stats['time_sec']:.1f}s"
# )

In [36]:
import os
checkpoint_dir = "/kaggle/working/lipsyncnet_checkpoints"
if os.path.exists(checkpoint_dir):
    for f in sorted(os.listdir(checkpoint_dir)):
        size_mb = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f"  {f:40s} {size_mb:.1f} MB")
else:
    print("Checkpoint directory does not exist")

Checkpoint directory does not exist


In [37]:
!pip install -q huggingface_hub

In [38]:
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO = "ranro1/lipsyncnet-checkpoints"

hf_api = HfApi(token=HF_TOKEN)

# Download checkpoint_epoch_20 from HF
checkpoint_path = hf_hub_download(
    repo_id=HF_REPO,
    filename="checkpoint_epoch_60.pt",
    token=HF_TOKEN,
    local_dir="/kaggle/working/lipsyncnet_checkpoints"
)
print(f"Downloaded to: {checkpoint_path}")

checkpoint_epoch_60.pt:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Downloaded to: /kaggle/working/lipsyncnet_checkpoints/checkpoint_epoch_60.pt


In [ ]:
def save_checkpoint_safe(model, optimizer, epoch, train_loss, val_loss, save_path,
                         push_to_hf=False):
    """Save locally always; optionally push to HF Hub."""
    tmp_path = "/tmp/checkpoint_tmp.pt"
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
    }
    torch.save(checkpoint, tmp_path)
    shutil.copy2(tmp_path, save_path)

    if push_to_hf:
        try:
            filename = os.path.basename(save_path)
            hf_api.upload_file(
                path_or_fileobj=tmp_path,
                path_in_repo=filename,
                repo_id="ranro1/lipsyncnet-checkpoints",
                repo_type="model",
                commit_message=f"epoch {epoch} | val_loss={val_loss:.4f}"
            )
            print(f"pushed {filename} to HF")
        except Exception as e:
            print(f"HF upload failed (non-fatal): {e}")


# ============================================================
# 100-EPOCH TRAINING
# ============================================================

checkpoint_dir = "/kaggle/working/lipsyncnet_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
history_path = os.path.join(checkpoint_dir, "training_history.pt")


# --- FRESH START ---
# paper = build_paper_model(device=device)
# optimizer = torch.optim.Adam(
#     [p for p in paper.parameters() if p.requires_grad],
#     lr=LEARNING_RATE
# )
# start_epoch = 0
# best_val_loss = float("inf")
# history = []

# --- OR RESUME (uncomment this block, comment out FRESH START above) ---
paper = build_paper_model(device=device)
optimizer = torch.optim.Adam(
    [p for p in paper.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)
paper, optimizer, start_epoch, _, _ = load_checkpoint(
    paper, optimizer,
    os.path.join(checkpoint_dir, "checkpoint_epoch_60.pt"),
    device
)
history = torch.load(history_path, map_location="cpu") if os.path.exists(history_path) else []
best_val_loss = min(h["val_loss"] for h in history) if history else float("inf")
print(f"Resuming from epoch {start_epoch + 1}, best_val_loss={best_val_loss:.4f}")

MAX_GRAD_NORM = 5.0
NUM_EPOCHS = 100

for epoch in range(start_epoch + 1, NUM_EPOCHS + 1):
    train_stats = train_one_epoch(
        model=paper,
        loader=train_loader,
        optimizer=optimizer,
        loss_fn=ctc_loss_fn,
        device=device,
        max_grad_norm=MAX_GRAD_NORM
    )

    val_stats = validate_one_epoch(
        model=paper,
        loader=val_loader,
        loss_fn=ctc_loss_fn,
        device=device
    )

    train_loss = train_stats["loss"]
    val_loss = val_stats["loss"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "grad_norm": train_stats["grad_norm"],
        "train_time": train_stats["time_sec"],
        "val_time": val_stats["time_sec"]
    })

    # Last checkpoint — overwritten every epoch (local only, fast)
    save_checkpoint_safe(
        model=paper,
        optimizer=optimizer,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        save_path=os.path.join(checkpoint_dir, "last_checkpoint.pt"),
        push_to_hf=False
    )

    # Best model — local only
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint_safe(
            model=paper,
            optimizer=optimizer,
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_loss,
            save_path=os.path.join(checkpoint_dir, "best_model.pt"),
            push_to_hf=False
        )
        print(f"  ↑ new best val_loss={val_loss:.4f} — saved best_model.pt")

    # Periodic checkpoint every 10 epochs — ALSO push to HF
    if epoch % 10 == 0:
        save_checkpoint_safe(
            model=paper,
            optimizer=optimizer,
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_loss,
            save_path=os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pt"),
            push_to_hf=True
        )
        print(f"  → saved checkpoint_epoch_{epoch}.pt")

print(f"\nTraining complete. Best val_loss={best_val_loss:.4f}")

Loaded checkpoint: epoch=60, val_loss=2.5529
Resuming from epoch 61, best_val_loss=inf


  ↑ new best val_loss=2.5282 — saved best_model.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed checkpoint_epoch_70.pt to HF
  → saved checkpoint_epoch_70.pt


Training:  16%|█▋        | 37/225 [00:39<03:26,  1.10s/it]   

In [ ]:
# from huggingface_hub import hf_hub_download

# # Download last checkpoint from HF
# checkpoint_path = hf_hub_download(
#     repo_id=HF_REPO,
#     filename="last_checkpoint.pt",
#     token=HF_TOKEN,
#     local_dir="/kaggle/working/lipsyncnet_checkpoints"
# )

# # Then load and resume as usual
# paper, optimizer, start_epoch, _, _ = load_checkpoint(
#     paper, optimizer,
#     checkpoint_path,
#     device
# )

## TODO: EVALUATION

## TODO-EVL-1  [CTC decoder]
- Paper uses beam search with beam_width=100. Implement both:
  (a) Greedy (argmax) — fast, use during training for monitoring.
  (b) Beam search (beam_width=100) — use for all reported results.
- Options: `torchaudio.functional.ctc_decode` (>=0.12) or `pip install ctcdecode`(faster, widely used in lip-reading papers).
- State which implementation was used in all result tables.

## TODO-EVL-2  [WER computation]
- `pip install jiwer`: Compute on the full test split after each run completes:
  `import jiwer` -> `wer = jiwer.wer(reference_list, hypothesis_list)`
- GRID: report both WER and word accuracy (= 1 − WER); paper uses accuracy.
- LRS2: report WER only (standard for sentence-level benchmarks).

## TODO-EVL-3  [Baseline comparison table — GRID]
- Build a table reproducing and extending paper Table 5:
  - Xu et al. [29]- - GRID  89.6% - Cascaded Attention-CTC
  - Gergen et al. [31]  GRID  86.4%
  - Margam et al. [30]  GRID  91.4% - 3D-2D-CNN BLSTM-HMM
  - LipSyncNet (ours) - GRID  target: 96.7% acc / 8.2% WER
- Add rows for each of the three temporal variants on both GRID-1K (paper subset) and GRID-full (speaker-independent split).
- Label the two conditions clearly in the table header.

## TODO-EVL-4  [Baseline comparison table — LRS2]
- Include established LRS2 test-set baselines: LipNet (Assael et al., 2016) and TM-seq2seq (Afouras et al.)
- Report all four model conditions on the official LRS2 test split.

## TODO-EVL-5  [Inference timing]
- Measure ms/sample (single clip, batch=1, GPU, torch.no_grad()) for all four model conditions.  Include in the variant comparison table, as it quantifies the cost-benefit of each temporal backend.

## TODO-EVL-6  [Variant ablation table — capstone report]
- Unified table with columns:
  Model | Backend | GRID-1K Acc | GRID-full Acc | LRS2 WER | Params | ms/sample
- Identity backend gives the "no temporal modeling" floor.
- Report honestly: if any variant underperforms Identity, investigate and explain why.

## TODO-EVL-7  [Parameter count audit table — named section in capstone]
- Run count_parameters() on all four models.
  Table columns:
  Model | Total | Trainable | Frozen | Notes
- In Notes: reference the noted discrepancies explicitly. Do NOT omit or round counts to make them match the paper's stated values.

## TODO-EVL-8  [Qualitative error analysis]
- Collect the 20 most-confused word pairs from GRID-full test predictions.
- Check whether they correspond to known viseme confusions (p/b, f/v, m/b).
- Include a short discussion in the capstone.